# AF2 controlled illumination robustness — seed-42 screening
Inference-only D0FT vs AF2 pada validation yang sama. Tidak training dan tidak membuka test.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import importlib, json, os, shutil, subprocess, sys, tarfile, time, torch
from pathlib import Path
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
REPO=Path('/content/coffee-bean-detection'); BRANCH='agent/af2-illumination-robustness'
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
from coffee_detector.drive_project import resolve_drive_project_root, require_project_artifact
REQ=('bundles/faruq-development-v3-grouped.tar','experiments/faruq-v3-acmc-optimization-control-v1/D0FT_seed42/weights/best.pt','experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt')
PROJECT=resolve_drive_project_root(required_relative_paths=REQ); ARCHIVE=require_project_artifact(PROJECT,REQ[0])
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'faruq_grouped_summary.json').is_file():
    with tarfile.open(ARCHIVE,'r') as archive:
        members=[m for m in archive.getmembers() if m.name.endswith('/data.yaml') or m.name.endswith('/faruq_grouped_summary.json') or '/val/' in m.name]
        archive.extractall('/content',members=members,filter='data')
assert (DATA/'val/images').is_dir() and not (DATA/'test').exists()
OUTPUT=PROJECT/'experiments/faruq-v3-af2-illumination-v1'
print('GPU:',torch.cuda.get_device_name(0)); print('PROJECT:',PROJECT); print('OUTPUT:',OUTPUT)


In [ ]:
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_illumination','--project-root',str(PROJECT),'--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),'--output-root',str(OUTPUT),'--seeds','42','--device','0']
log=OUTPUT/'screen_seed42_run.log'; log.parent.mkdir(parents=True,exist_ok=True)
print('MENJALANKAN:', ' '.join(command),flush=True)
handle=log.open('a',encoding='utf-8'); process=subprocess.Popen(command,cwd=REPO,stdout=handle,stderr=subprocess.STDOUT,text=True)
while process.poll() is None:
    reports=len(list((OUTPUT/'reports/seed42').rglob('*.json'))) if (OUTPUT/'reports/seed42').exists() else 0
    print(f'ILLUMINATION SCREEN: {reports}/20 evaluasi selesai | log={log}',flush=True); time.sleep(300)
handle.close(); code=process.wait()
if code: print('\n'.join(log.read_text(errors='replace').splitlines()[-120:])); raise RuntimeError(f'Screening gagal: {code}')


In [ ]:
import pandas as pd
from IPython.display import Image as DisplayImage, display
summary=json.loads((OUTPUT/'illumination_screen_seed42.json').read_text())
display(pd.DataFrame(summary['aggregate']).style.format({'mean_robustness_advantage':'{:+.2%}','minimum_robustness_advantage':'{:+.2%}'}))
display(pd.DataFrame(summary['effects'])[['condition','family','robustness_advantage_macro_map50_95','robustness_advantage_bottom3_class_map50_95','robustness_advantage_worst_class_map50_95']].style.format({c:'{:+.2%}' for c in ['robustness_advantage_macro_map50_95','robustness_advantage_bottom3_class_map50_95','robustness_advantage_worst_class_map50_95']}))
display(DisplayImage(filename=str(OUTPUT/'illumination_preview.jpg')))
print('CRITERIA:',summary['criteria']); print('DECISION:',summary['decision']); print('CONFIRMATION AUTHORIZED:',summary['confirmation_authorized']); print('TEST:',summary['test_images_accessed'])
print('Kirim dua tabel, preview, criteria, dan decision. Jangan jalankan tiga seed jika FAIL.')
